In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import json
from datetime import datetime
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from PIL import Image as PILImage

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PROCESSED_DIR = Path(r"C:\Users\MSI GF66\OneDrive - National Polyechnic University of Armenia\Desktop\Emotion_recognition\data\processed\emotions")
MODELS_DIR = Path(r"C:\Users\MSI GF66\OneDrive - National Polyechnic University of Armenia\Desktop\Emotion_recognition\models")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path(r"C:\Users\MSI GF66\OneDrive - National Polyechnic University of Armenia\Desktop\Emotion_recognition\results\enhanced_ensemble_1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 0.001
NUM_MODELS = 5

X_train = np.load(PROCESSED_DIR / 'X_train.npy')
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
X_val = np.load(PROCESSED_DIR / 'X_val.npy')
y_val = np.load(PROCESSED_DIR / 'y_val.npy')
X_test = np.load(PROCESSED_DIR / 'X_test.npy')
y_test = np.load(PROCESSED_DIR / 'y_test.npy')

with open(PROCESSED_DIR / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

EMOTION_LABELS = metadata['class_labels']
NUM_CLASSES = len(EMOTION_LABELS)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")


class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        image = (image * 255).astype(np.uint8).squeeze()
        image = PILImage.fromarray(image, mode='L')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = EmotionDataset(X_train, y_train, transform=train_transform)
val_dataset = EmotionDataset(X_val, y_val, transform=val_transform)
test_dataset = EmotionDataset(X_test, y_test, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)

        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.skip(x)
        out = F.relu(out)
        return out


class EmotionNet(nn.Module):
    def __init__(self, num_classes=7, dropout=0.5):
        super(EmotionNet, self).__init__()

        self.conv1 = nn.Conv2d(1, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(512, 512)
        self.bn_fc1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(512, 256)
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout)

        self.fc3 = nn.Linear(256, num_classes)

    def _make_layer(self, in_channels, out_channels, num_blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = self.bn_fc1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn_fc2(x)
        x = F.relu(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        return x


def train_single_model(model_id, train_loader, val_loader, device, class_weights, num_classes, epochs):
    print(f"\nTraining Model {model_id + 1}/{NUM_MODELS}")
    print("=" * 70)

    seed = 42 + model_id * 100
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model = EmotionNet(num_classes=num_classes, dropout=0.5).to(device)

    criterion = FocalLoss(alpha=class_weights, gamma=2.5)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LEARNING_RATE * 2,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )
    scaler = GradScaler()

    best_val_acc = 0.0
    patience = 15
    patience_counter = 0
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        history.append({'train_acc': train_acc, 'val_acc': val_acc})

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch+1}: Train={train_acc*100:.2f}% | Val={val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODELS_DIR / f'enhanced_model_{model_id}.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(MODELS_DIR / f'enhanced_model_{model_id}.pth'))
    print(f"Model {model_id + 1} best val accuracy: {best_val_acc*100:.2f}%")

    return model, best_val_acc, history


class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.FloatTensor(class_weights).to(device)

print("\nTraining ensemble...")
print(f"Class weights: {class_weights.cpu().numpy()}")

start_time = datetime.now()

models_list = []
val_accs = []
all_histories = []

for i in range(NUM_MODELS):
    model, best_val_acc, history = train_single_model(
        i, train_loader, val_loader, device, class_weights, NUM_CLASSES, EPOCHS
    )
    models_list.append(model)
    val_accs.append(best_val_acc)
    all_histories.append(history)

training_time = (datetime.now() - start_time).total_seconds() / 60
print(f"\nTotal training time: {training_time:.1f} minutes")
print(f"Average val accuracy: {np.mean(val_accs)*100:.2f}%")
print(f"Std val accuracy: {np.std(val_accs)*100:.2f}%")


def ensemble_predict_voting(models_list, loader, device):
    all_votes = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Ensemble Voting'):
        images = images.to(device)
        batch_votes = []

        for model in models_list:
            model.eval()
            with torch.no_grad():
                with autocast():
                    outputs = model(images)
                    _, predicted = outputs.max(1)
                    batch_votes.append(predicted.cpu().numpy())

        batch_votes = np.array(batch_votes)

        final_predictions = []
        for i in range(batch_votes.shape[1]):
            votes = batch_votes[:, i]
            prediction = np.bincount(votes).argmax()
            final_predictions.append(prediction)

        all_votes.extend(final_predictions)
        all_labels.extend(labels.numpy())

    return np.array(all_votes), np.array(all_labels)


def ensemble_predict_weighted(models_list, loader, device, weights):
    all_outputs = []
    all_labels = []

    for model in models_list:
        model.eval()
        model_outputs = []

        with torch.no_grad():
            for images, labels in tqdm(loader, desc='Weighted Ensemble', leave=False):
                images = images.to(device)

                with autocast():
                    outputs = model(images)
                    probs = F.softmax(outputs, dim=1)

                model_outputs.append(probs)

                if len(all_labels) == 0:
                    all_labels.append(labels)

        all_outputs.append(torch.cat(model_outputs))

        if len(all_labels) == 1:
            all_labels = all_labels[0]

    weighted_probs = sum(w * out for w, out in zip(weights, all_outputs))
    _, predictions = weighted_probs.max(1)

    return predictions.cpu().numpy(), all_labels.numpy()


print("\nEvaluating individual models on test set...")
individual_preds = []
individual_accs = []

for i, model in enumerate(models_list):
    model.eval()
    preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            preds.extend(predicted.cpu().numpy())

    preds = np.array(preds)
    individual_preds.append(preds)
    acc = (preds == y_test).mean()
    individual_accs.append(acc)
    print(f"Model {i+1}: {acc*100:.2f}%")

print(f"\nAverage individual: {np.mean(individual_accs)*100:.2f}%")
print(f"Best individual: {max(individual_accs)*100:.2f}%")

print("\nMethod 1: Majority Voting...")
voting_preds, test_labels = ensemble_predict_voting(models_list, test_loader, device)
voting_acc = (voting_preds == test_labels).mean()
print(f"Voting Accuracy: {voting_acc*100:.2f}%")

print("\nMethod 2: Weighted Average (by validation accuracy)...")
weights = torch.tensor([acc / sum(val_accs) for acc in val_accs])
weighted_preds, _ = ensemble_predict_weighted(models_list, test_loader, device, weights)
weighted_acc = (weighted_preds == test_labels).mean()
print(f"Weighted Accuracy: {weighted_acc*100:.2f}%")

best_method = 'voting' if voting_acc > weighted_acc else 'weighted'
best_preds = voting_preds if voting_acc > weighted_acc else weighted_preds
best_acc = max(voting_acc, weighted_acc)

print(f"\nBest method: {best_method.upper()}")
print(f"Best ensemble accuracy: {best_acc*100:.2f}%")
print(f"Improvement over best individual: +{(best_acc - max(individual_accs))*100:.2f}%")

print("\n" + classification_report(test_labels, best_preds, target_names=EMOTION_LABELS, digits=4))

cm = confusion_matrix(test_labels, best_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[e.capitalize() for e in EMOTION_LABELS],
            yticklabels=[e.capitalize() for e in EMOTION_LABELS],
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Ensemble Confusion Matrix - {best_method.capitalize()}')

sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=[e.capitalize() for e in EMOTION_LABELS],
            yticklabels=[e.capitalize() for e in EMOTION_LABELS],
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Normalized Confusion Matrix')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ensemble_confusion_matrix.png', dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(models_list) + 3)
accuracies = individual_accs + [np.mean(individual_accs), voting_acc, weighted_acc]
labels = [f'M{i+1}' for i in range(len(models_list))] + ['Avg', 'Vote', 'Weighted']
colors = ['skyblue'] * len(models_list) + ['orange', 'green', 'red']

bars = ax.bar(x_pos, [a*100 for a in accuracies], color=colors, alpha=0.8)
ax.set_xlabel('Models')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Individual vs Ensemble Performance')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.axhline(y=best_acc*100, color='darkred', linestyle='--', linewidth=2, label=f'Best: {best_acc*100:.2f}%')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{acc*100:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ensemble_comparison.png', dpi=300)
plt.show()

per_class_acc = cm.diagonal() / cm.sum(axis=1)
performance_df = pd.DataFrame({
    'Emotion': [e.capitalize() for e in EMOTION_LABELS],
    'Accuracy': per_class_acc
}).sort_values('Accuracy', ascending=False)

print("\nPer-Class Accuracy:")
print(performance_df.to_string(index=False))

model_info = {
    'model_name': 'Enhanced Ensemble EmotionNet',
    'architecture': 'ResNet + SE + Focal Loss',
    'num_models': NUM_MODELS,
    'enhancements': [
        'Focal Loss (gamma=2.5)',
        'SE attention blocks',
        'OneCycleLR with warmup',
        'Gradient clipping',
        'Weighted ensemble',
        'Voting ensemble'
    ],
    'training': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'training_time_minutes': float(training_time),
    },
    'performance': {
        'individual_accs': [float(acc) for acc in individual_accs],
        'voting_acc': float(voting_acc),
        'weighted_acc': float(weighted_acc),
        'best_ensemble_acc': float(best_acc),
        'improvement_over_best': float(best_acc - max(individual_accs)),
        'per_class_accuracy': {EMOTION_LABELS[i]: float(per_class_acc[i]) for i in range(NUM_CLASSES)}
    },
    'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open(RESULTS_DIR / 'model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print(f"\nFinal Ensemble Accuracy: {best_acc*100:.2f}%")
print(f"Training Time: {training_time:.1f} minutes")
print(f"Method: {best_method}")
print(f"Models saved to: {MODELS_DIR}")
print(f"Results saved to: {RESULTS_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU Memory: 6.00 GB

Memory-optimized settings:
Batch size: 32
Epochs: 60
Number of models: 3 (reduced for memory)

Train: (24402, 48, 48, 1)
Val: (4307, 48, 48, 1)
Test: (7178, 48, 48, 1)

TRAINING ENSEMBLE MODELS SEQUENTIALLY

Training Model 1/3
Parameters: 11,652,039


Epoch 1/60:  33%|███▎      | 249/763 [00:14<00:28, 18.28it/s, loss=1.8365, acc=16.69%]